[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [APIs and JSON](https://johnfisher-ai.github.io/Python-Visual-Guides/apis-and-json.html)

# Schemas and Validation &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The cell below rebuilds what the notebook set up, including the practice API, `station_schema`,
`documented_station` and the Pydantic models. Run it first.


In [1]:
import importlib
import sys
import urllib.request
from datetime import date, datetime
from pathlib import Path

import requests
from jsonschema import Draft202012Validator, FormatChecker
from pydantic import BaseModel, Field

PRACTICE_API = "https://raw.githubusercontent.com/johnfisher-ai/Python-Visual-Guides/main/notebooks/apis-and-json/practice_api.py"

if "google.colab" in sys.modules or not Path("practice_api.py").exists():
    urllib.request.urlretrieve(PRACTICE_API, "practice_api.py")    # in Colab, on every run

import practice_api
importlib.reload(practice_api)    # runs the file as it is now, not a copy imported earlier

BASE = practice_api.start()

location_schema = {"type": "object", "required": ["latitude", "longitude"],
                   "properties": {"latitude": {"type": "number", "minimum": -90, "maximum": 90},
                                  "longitude": {"type": "number", "minimum": -180, "maximum": 180},
                                  "elevation_m": {"type": "integer"}}}
instrument_schema = {"type": "object", "required": ["kind", "installed", "last_calibrated"],
                     "properties": {"kind": {"type": "string"},
                                    "installed": {"type": "string", "format": "date"},
                                    "last_calibrated": {"type": ["string", "null"], "format": "date"}}}
status_schema = {"type": ["object", "null"], "required": ["active", "issues"],
                 "properties": {"active": {"type": "boolean"}, "issues": {"type": "array"}}}
station_schema = {"type": "object", "required": ["id", "name", "location", "instruments", "status"],
                  "properties": {"id": {"type": "string"}, "name": {"type": "string"},
                                 "location": location_schema,
                                 "instruments": {"type": "array", "items": instrument_schema},
                                 "status": status_schema}}

documented_station = requests.get(f"{BASE}/openapi.json", timeout=10).json()["components"]["schemas"]["Station"]


class Location(BaseModel):
    latitude: float = Field(ge=-90, le=90)
    longitude: float = Field(ge=-180, le=180)
    elevation_m: int | None = None


class Instrument(BaseModel):
    kind: str
    installed: date
    last_calibrated: date | None


class Issue(BaseModel):
    since: date
    summary: str


class Status(BaseModel):
    active: bool
    issues: list[Issue]


class Station(BaseModel):
    id: str
    name: str
    location: Location
    instruments: list[Instrument]
    status: Status | None


class Network(BaseModel):
    name: str
    updated: datetime
    stations: list[Station]


print("ready:", BASE)


ready: http://127.0.0.1:8765


**1.** A response checked against its own documentation.


In [2]:
svalbard = requests.get(f"{BASE}/stations/svalbard", timeout=10).json()

print(len(list(Draft202012Validator(documented_station).iter_errors(svalbard))), "errors")


0 errors


`iter_errors` yields nothing for data that matches, so the list is empty and its length is `0`.


**2.** A made-up station, checked.


In [3]:
bodo = {"id": "bodo", "name": "Bodo", "latitude": 67.28}

for error in Draft202012Validator(documented_station).iter_errors(bodo):
    print(error.json_path, "|", error.message)


$ | 'longitude' is a required property


The path is `$`, the station itself, because a missing member is an error of the object that should
hold it.


**3.** A schema that forbids members it does not mention.


In [4]:
closed = {**station_schema, "additionalProperties": False}
tromso = requests.get(f"{BASE}/beta/network", timeout=10).json()["stations"][3]

for error in Draft202012Validator(closed, format_checker=FormatChecker()).iter_errors(tromso):
    print(error.message)


'id' is a required property
Additional properties are not allowed ('station_id' was unexpected)


A renamed member shows up twice in a closed schema: once as the member that is missing, and once as
the member nobody expected. A schema that allows extra members reports only the first.


**4.** A model for a list's items.


In [5]:
class Summary(BaseModel):
    id: str
    name: str


summaries = [Summary.model_validate(item) for item in requests.get(f"{BASE}/stations", timeout=10).json()]
print([summary.name for summary in summaries])


['Bergen', 'Oslo', 'Svalbard', 'Tromso']


The response is a list, so each item is validated on its own. For the whole list in one call,
Pydantic's `TypeAdapter(list[Summary]).validate_python(items)` returns the same list of objects.


**5.** Attributes, and dates that compare as dates.


In [6]:
checked = Network.model_validate_json(requests.get(f"{BASE}/network", timeout=10).content, strict=True)

print([station.name for station in checked.stations if station.status is None])
print([instrument.kind for station in checked.stations for instrument in station.instruments
       if instrument.last_calibrated is not None and instrument.last_calibrated < date(2025, 1, 1)])


['Tromso']
['anemometer']


`last_calibrated` is a `date` or `None`, as the model declared, so the `None` check comes first and
the comparison needs no parsing.


**6.** The members a model does not see.


In [7]:
beta_stations = requests.get(f"{BASE}/beta/network", timeout=10).json()["stations"]

print({member for item in beta_stations for member in item} - set(Station.model_fields))


{'station_id'}


`model_fields` maps each declared member's name to its definition, so a set of its keys is the set of
declared names. Checking for new members this way tells you what a response has started to send
before any code relies on it.


---

&#8592; **Back to:** [Schemas and Validation](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/apis-and-json/07-schemas-and-validation.ipynb)  &nbsp;&middot;&nbsp;  [APIs and JSON Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/apis-and-json.html)
